In [4]:
import time
import logging
import requests
import xml.etree.ElementTree as ET
from sina.config.credentials import cne_localidades_url, cne_precios_gas_lp_url
from sina.db.repository import get_session, LocalidadRepository, MunicipioRepository
from sina.db.models import Localidad, Municipio
logger = logging.getLogger(__name__)
_NOMBRES_SKIP = {"ninguno", "none", ""}

In [8]:
url = cne_localidades_url
params = {
    "entidadFederativaId": 26,
    "municipioId": "030",
}

resp = requests.get(url, params=params, timeout=15)

# ── Veamos qué está llegando exactamente ──
print("Status code:", resp.status_code)
print("Content-Type:", resp.headers.get("Content-Type"))
print("Encoding:", resp.encoding)
print("Tamaño (bytes):", len(resp.content))
print("\n── Primeros 500 chars del contenido ──")
print(repr(resp.content[:500]))  # repr para ver bytes exactos, BOM, etc.

Status code: 200
Content-Type: application/json; charset=utf-8
Encoding: utf-8
Tamaño (bytes): 436656

── Primeros 500 chars del contenido ──
b'[{"Id":664,"Nombre":"20 de Noviembre","EntidadFederativa":{"EntidadFederativaId":"26","Nombre":"Sonora"},"EntidadFederativaId":"26","Municipio":{"MunicipioId":"030","EntidadFederativaId":"26","Nombre":"Hermosillo","EntidadFederativa":null},"MunicipioId":"030"},{"Id":3887,"Nombre":"20 de Noviembre Dos","EntidadFederativa":{"EntidadFederativaId":"26","Nombre":"Sonora"},"EntidadFederativaId":"26","Municipio":{"MunicipioId":"030","EntidadFederativaId":"26","Nombre":"Hermosillo","EntidadFederativa":n'


In [9]:
# Reemplazar _parsear_localidades_xml por esto:

def _parsear_localidades_json(content: bytes) -> list[dict]:
    """
    Parsea el JSON de respuesta de la API de localidades.

    Estructura real:
    [
        {
            "Id": 664,
            "Nombre": "20 de Noviembre",
            "EntidadFederativaId": "26",
            "EntidadFederativa": {"EntidadFederativaId": "26", "Nombre": "Sonora"},
            "MunicipioId": "030",
            "Municipio": {"MunicipioId": "030", "Nombre": "Hermosillo", ...}
        },
        ...
    ]
    """
    import json

    try:
        data = json.loads(content)
    except json.JSONDecodeError as e:
        logger.error(f"Error parseando JSON de localidades: {e}")
        return []

    if not isinstance(data, list):
        logger.error(f"Se esperaba una lista, llegó: {type(data)}")
        return []

    resultados = []

    for item in data:
        # ── ID ────────────────────────────────────────────────
        loc_id = item.get("Id")

        if loc_id is None:
            continue

        # Asegurarnos que sea entero válido
        try:
            loc_id = int(loc_id)
        except (ValueError, TypeError):
            continue

        # ── Nombre ───────────────────────────────────────────
        nombre = item.get("Nombre") or ""
        nombre = nombre.strip()

        # Skip vacíos o "Ninguno" (incluyendo "Ninguno [Sergio Ruiz Montaño]")
        if not nombre or nombre.lower().startswith("ninguno"):
            logger.debug(f"  Skipping id={loc_id} nombre='{nombre}'")
            continue

        resultados.append({
            "localidad_id": loc_id,
            "nombre":       nombre,
        })

    return resultados

In [10]:
def fetch_localidades(entidad_id: int, municipio_id_str: str) -> list[dict]:
    url = cne_localidades_url
    params = {
        "entidadFederativaId": entidad_id,
        "municipioId":         municipio_id_str,
    }

    try:
        resp = requests.get(url, params=params, timeout=15)
        resp.raise_for_status()
    except requests.RequestException as e:
        logger.error(f"Error al pedir localidades (entidad={entidad_id}, mun={municipio_id_str}): {e}")
        return []

    # ← JSON, no XML
    return _parsear_localidades_json(resp.content)

In [13]:
localidades_hermosillo = fetch_localidades(26, "030") 

In [14]:
localidades_hermosillo

[{'localidad_id': 664, 'nombre': '20 de Noviembre'},
 {'localidad_id': 3887, 'nombre': '20 de Noviembre Dos'},
 {'localidad_id': 3888, 'nombre': '20 de Noviembre Tres'},
 {'localidad_id': 3943, 'nombre': '20 Noviembre Cuatro'},
 {'localidad_id': 1149, 'nombre': '6 de Diciembre (El Veintiocho)'},
 {'localidad_id': 3659, 'nombre': 'Acuícola México'},
 {'localidad_id': 3660, 'nombre': 'Acuícola Polo'},
 {'localidad_id': 3661, 'nombre': 'Acuícola Selecta'},
 {'localidad_id': 3925, 'nombre': 'Acuícola Tastiota'},
 {'localidad_id': 3544, 'nombre': 'Adalberto Córdova'},
 {'localidad_id': 2934, 'nombre': 'Adán López'},
 {'localidad_id': 2935, 'nombre': 'Aeropuerto Bahía de Kino'},
 {'localidad_id': 1055, 'nombre': 'Agrícola Doña Lola (Las Playitas)'},
 {'localidad_id': 3963, 'nombre': 'Agrícola Martha'},
 {'localidad_id': 719, 'nombre': 'Agrícola Oremor (Santa Patricia)'},
 {'localidad_id': 2897, 'nombre': 'Agrícola Río Blanco'},
 {'localidad_id': 3831, 'nombre': 'Agrobal 6'},
 {'localidad_id'

In [12]:
# Celda 1 — imports
from sina.db.repository import get_session, MunicipioRepository
from sina.db.models import Localidad, Municipio

# Celda 2 — obtener la PK del municipio en nuestra DB
# (necesitamos el id de cne_municipios, NO el municipio_id string)
with get_session() as session:
    mun = (
        session.query(Municipio)
        .filter_by(municipio_id="030", entidad_id=26)
        .first()
    )
    print(f"Municipio encontrado: {mun}")
    print(f"  PK (id):          {mun.id}")
    print(f"  municipio_id:     {mun.municipio_id}")
    print(f"  entidad_id:       {mun.entidad_id}")
    print(f"  nombre:           {mun.nombre}")
    municipio_pk = mun.id  # ← esta es la FK que necesita Localidad

Municipio encontrado: <sina.db.models.Municipio object at 0x0000023AF227D4F0>
  PK (id):          1938
  municipio_id:     030
  entidad_id:       26
  nombre:           Hermosillo


In [15]:
insertadas = 0
skipped    = 0

with get_session() as session:
    for loc in localidades_hermosillo:
        # Verificar si ya existe
        existe = (
            session.query(Localidad)
            .filter_by(
                localidad_id=loc["localidad_id"],
                municipio_id=municipio_pk,
            )
            .first()
        )

        if existe is None:
            nueva = Localidad(
                localidad_id=loc["localidad_id"],
                nombre=loc["nombre"],
                municipio_id=municipio_pk,
            )
            session.add(nueva)
            insertadas += 1
        else:
            skipped += 1

    session.commit()

print(f"✅ Insertadas: {insertadas}")
print(f"⏭️  Ya existían: {skipped}")

✅ Insertadas: 1621
⏭️  Ya existían: 0


In [16]:
# Celda 4 — verificar que quedaron bien
with get_session() as session:
    locs = (
        session.query(Localidad)
        .filter_by(municipio_id=municipio_pk)
        .order_by(Localidad.nombre)
        .all()
    )
    print(f"Total localidades en DB para Hermosillo: {len(locs)}")
    for l in locs[:10]:
        print(f"  id={l.localidad_id:>6} | {l.nombre}")

Total localidades en DB para Hermosillo: 1621
  id=  3943 | 20 Noviembre Cuatro
  id=   664 | 20 de Noviembre
  id=  3887 | 20 de Noviembre Dos
  id=  3888 | 20 de Noviembre Tres
  id=  1149 | 6 de Diciembre (El Veintiocho)
  id=  3310 | AMSA (TIASA)
  id=  3659 | Acuícola México
  id=  3660 | Acuícola Polo
  id=  3661 | Acuícola Selecta
  id=  3925 | Acuícola Tastiota
